# Phase 2: Generative AI & Prompt Engineering
This notebook is dedicated to exploring and refining prompts using groq API.

# 1-API Setup 


In [ ]:
# INSTALL DEPENDENCIES
%pip install groq python-dotenv


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# IMPORTS libraries 
import os
import pandas as pd
import sys
sys.path.append("Generative_AI") # Tell Python where to find api_config.py 
from api_config import client          
import warnings
warnings.filterwarnings("ignore")

In [6]:
# Test connection to Groq
try:
    test = client.chat.completions.create(
        model="meta-llama/llama-4-scout-17b-16e-instruct",
        messages=[{"role": "user", "content": "Reply with OK only."}],
        max_completion_tokens=10 
    )
    print("Groq connected:", test.choices[0].message.content)
except Exception as e:
    print("Connection failed:", e)


Groq connected: OK


# 2- Prompt Function 


In [ ]:
#   Sends a prompt to Groq (Llama) and returns the generated advice.
#    temperature=0 keeps responses consistent across all templates.

def generate_attrition_advice(prompt, temperature=0): 
    try:
        response = client.chat.completions.create(
            model="meta-llama/llama-4-scout-17b-16e-instruct",
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature,
            max_completion_tokens=500
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        return f"Error: {e}"


# 3-LOAD DATASET


In [10]:
# Load the preprocessed dataset 
df = pd.read_csv("Dataset/preprocessed_data.csv")
print("Shape:", df.shape)
print(df.head())

Shape: (1470, 54)
        Age  Attrition  BusinessTravel  DailyRate  DistanceFromHome  \
0  0.446350          1               1   0.742527         -1.010909   
1  1.322365          0               2  -1.297775         -0.147150   
2  0.008343          1               1   1.414363         -0.887515   
3 -0.429664          0               2   1.461466         -0.764121   
4 -1.086676          0               1  -0.524295         -0.887515   

   Education  EnvironmentSatisfaction  Gender  HourlyRate  JobInvolvement  \
0          2                        2  Female    1.383138               3   
1          1                        3    Male   -0.240677               2   
2          2                        4    Male    1.284725               2   
3          4                        4  Female   -0.486709               3   
4          1                        1    Male   -1.274014               3   

   ...  JobRole_Sales Representative  MaritalStatus_Divorced  \
0  ...                      

# 4-TEST CASES 


In [11]:
# Features selected based on top coefficients from our Logistic Regression model.
# Feel free to change as needed based on what works best for the templates.

FEATURE_COLS = [
    "OverTime",                
    "BusinessTravel",          
    "YearsInCurrentRole",     
    "NumCompaniesWorked",      
    "MaritalStatus_Single",   
    "JobInvolvement",         
    "TotalWorkingYears",       
    "Age",                    
]


TARGET_COL = "Attrition"  # 1 = likely to leave, 0 = likely to stay

sampled = df[FEATURE_COLS + [TARGET_COL]].sample(3, random_state=42).reset_index(drop=True)
TEST_CASES = sampled.to_dict(orient="records")

for i, case in enumerate(TEST_CASES, 1):
    print(f"\nCase {i}: {case}")

print(f"\n{len(TEST_CASES)} test cases ready")


Case 1: {'OverTime': 'No', 'BusinessTravel': 1, 'YearsInCurrentRole': -0.063295899399149, 'NumCompaniesWorked': -1.0785044383346123, 'MaritalStatus_Single': True, 'JobInvolvement': 3, 'TotalWorkingYears': -0.6835478776036732, 'Age': -0.9771736638126856, 'Attrition': 0}

Case 2: {'OverTime': 'No', 'BusinessTravel': 1, 'YearsInCurrentRole': -0.6154915796172451, 'NumCompaniesWorked': -0.6780493930322936, 'MaritalStatus_Single': False, 'JobInvolvement': 4, 'TotalWorkingYears': -0.8132851765161844, 'Age': 1.7603726195472926, 'Attrition': 0}

Case 3: {'OverTime': 'No', 'BusinessTravel': 1, 'YearsInCurrentRole': -1.1676872598353414, 'NumCompaniesWorked': -0.6780493930322936, 'MaritalStatus_Single': False, 'JobInvolvement': 1, 'TotalWorkingYears': -1.3322343721662295, 'Age': -1.4151810691502822, 'Attrition': 1}

3 test cases ready


In [ ]:
# SAVE EXAMPLE OUTPUTS FCUNCTION
def save_example_output(template_name, case_id, response):
    """Saves a response to Generative_AI/example_outputs/"""
    os.makedirs("Generative_AI/example_outputs", exist_ok=True)
    filename = f"Generative_AI/example_outputs/{template_name}_case{case_id}.txt"
    with open(filename, "w") as f:
        f.write(response)
    print(f"Saved: {filename}")


# 5- PROMPT EXPERIMENTS

In [ ]:
# will be filled with the prompt templates 
TEMPLATES = {
    "template_1": "[Enter your first prompt design here. Use {case} where you want data to appear]",
    "template_2": "[Enter your second prompt design here. Use {case} where you want data to appear]",
    "template_3": "[Enter your third prompt design here. Use {case} where you want data to appear]"
}

# RUN EXPERIMENTS
print("Starting Prompt Testing")

for template_name, template_text in TEMPLATES.items():
    print(f"\n Running Template: {template_name} ")
    
    for i, case in enumerate(TEST_CASES, 1):
        # 1. Prepare the prompt by inserting the employee data
        full_prompt = template_text.format(case=case)
        
        # 2. Get the response from Groq
        advice = generate_attrition_advice(full_prompt)
        
        # 3. Save the result to the example_outputs folder
        save_example_output(template_name, i, advice)
        
        print(f" Case {i} saved.")

print("\nAll experiments completed! Check the 'Generative_AI/example_outputs' folder for results.")
